# Medical Dictation Transcription & SOAP Note Generator

**Pipeline:**
1. **Part A** — Transcribe audio using `faster-whisper` (local, CPU, no cloud API)
2. **Part B** — Generate structured SOAP note using NLP extraction (local, no cloud API)
3. **Bonus** — S/O contamination validation check

**Tool Choices:**
| Task | Tool | Reason |
|------|------|---------|
| Speech-to-Text | `faster-whisper` (medium model) | Local, 4x faster than OpenAI Whisper, handles medical terminology accurately |
| SOAP Generation | NLP extraction + structured prompt | Deterministic, no hallucination risk, fully offline |
| LLM (optional) | Ollama + llama3 | Local LLM fallback if available — patient data never leaves machine |

## Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'faster-whisper', 'ollama'], check=True)

In [ ]:
import os, json, re
from pathlib import Path

AUDIO_FILE  = "sample_dictation.mp3"
OUTPUT_DIR  = Path("output")
MODELS_DIR  = Path("models")
OLLAMA_MODEL = "llama3"

OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
print(f"Audio      : {AUDIO_FILE}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")

## Part A: Audio Transcription (faster-whisper)

**Why faster-whisper?**
- Runs 100% locally — no audio data sent to any server (HIPAA-friendly)
- 4x faster than original OpenAI Whisper with identical accuracy
- `medium` model handles medical terminology (epicondylitis, pronation, NSAIDs) accurately
- CTranslate2 backend optimized for CPU int8 inference

**Model download (first run only):**
```bash
python -c "
import huggingface_hub
for f in ['config.json','model.bin','tokenizer.json','vocabulary.txt']:
    huggingface_hub.hf_hub_download('Systran/faster-whisper-medium', f,
        local_dir='models/whisper-medium', local_dir_use_symlinks=False)
    print('Downloaded:', f)
"
```

In [ ]:
from faster_whisper import WhisperModel

def transcribe_audio(audio_path: str) -> str:
    """
    Transcribe audio locally using faster-whisper medium model.
    Model must be pre-downloaded to models/whisper-medium/ (see README).
    """
    model_path = str(Path("models/whisper-medium").resolve())
    print(f"Loading model from: {model_path}", flush=True)
    model = WhisperModel(model_path, device="cpu", compute_type="int8")

    print(f"Transcribing: {audio_path}", flush=True)
    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        language="en",
        condition_on_previous_text=True,  # improves coherence across segments
        vad_filter=True,                  # skip silent parts automatically
        vad_parameters=dict(min_silence_duration_ms=500)
    )
    print(f"Language detected: {info.language} (confidence: {info.language_probability:.2f})")

    parts = []
    for i, seg in enumerate(segments, 1):
        print(f"  [{i}] {seg.start:.1f}s -> {seg.end:.1f}s : {seg.text.strip()}", flush=True)
        parts.append(seg.text.strip())
    return " ".join(parts)


transcript = transcribe_audio(AUDIO_FILE)
(OUTPUT_DIR / "transcript.txt").write_text(transcript, encoding="utf-8")
print(f"\nTranscript saved to: {OUTPUT_DIR}/transcript.txt")
print(f"Total length: {len(transcript)} characters")

### Transcript Output

In [ ]:
print((OUTPUT_DIR / "transcript.txt").read_text(encoding="utf-8"))

what brings you in? Hi, I've had this pain on the outside of my right elbow now. I first started noticing it several months ago but recently it's just been more painful. Okay, so you said several months ago. Did anything happened several months ago? Was there any sort of trigger, trauma, anything like that to that area? No, there wasn't any trauma or any triggers that I noticed. I was just feeling it a bit more at the end of work. Yeah, I was just feeling the pain a bit more at the end of work. Okay, does anything make it better or worse, the pain? Yeah, really if I'm resting the elbow it makes it better and I've tried things like ibuprofen, which has helped with the pain. I'll do that for helping I get through work sometimes if the pain is bad enough. Right, okay and if you were to describe the quality of the pain, is it sharp, throbby, achy? It's kind of, well it's achy and then sometimes depending on the movement it can get, it can be sharp as well. It can be sharp? Okay, okay and w

## Part B: SOAP Note Generation

**Approach — Ollama (primary) with NLP extraction fallback:**
- If Ollama is running locally (`ollama pull llama3`), uses llama3 with a structured JSON prompt
- Falls back to deterministic NLP extraction from transcript if Ollama unavailable

**Prompt Engineering Strategy:**
- Explicit SOAP section definitions prevent S/O cross-contamination
- JSON schema enforced in prompt for structured output
- Temperature=0.1 for deterministic, factual output

In [ ]:
SOAP_SYSTEM_PROMPT = """You are a clinical documentation specialist converting a medical dictation into a SOAP note.

STRICT SECTION DEFINITIONS:
- Subjective (S): ONLY what the PATIENT reports — symptoms, pain, history as told by patient. Never include exam findings.
- Objective (O): ONLY what the CLINICIAN observes/measures — vitals, physical exam, labs/imaging. Never include patient-reported symptoms.
- Assessment (A): Clinician's diagnosis or clinical impression.
- Plan (P): Treatment, prescriptions, referrals, follow-up.

Return ONLY valid JSON. No explanation, no markdown."""

SOAP_USER_PROMPT = """Convert this transcript to a SOAP note JSON:

{transcript}

Use this exact structure (empty string if not mentioned):
{{
  "Subjective": {{"Chief Complaint":"","History of Present Illness":"","Pain Scale":"","Past Medical History":"","Current Medications":"","Allergies":"","Review of Systems":""}},
  "Objective": {{"Vital Signs":"","Physical Examination":"","Diagnostic Results":""}},
  "Assessment": {{"Primary Diagnosis":"","Differential Diagnosis":""}},
  "Plan": {{"Treatment":"","Medications Prescribed":"","Follow Up":"","Patient Education":""}}
}}"""


def extract_json(text: str) -> dict:
    text = re.sub(r"```(?:json)?\n?", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"No JSON found in: {text[:200]}")


def generate_soap_ollama(transcript: str) -> dict:
    """Primary: local Ollama LLM. Run: ollama pull llama3"""
    import ollama
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SOAP_SYSTEM_PROMPT},
            {"role": "user",   "content": SOAP_USER_PROMPT.format(transcript=transcript)}
        ],
        options={"temperature": 0.1}
    )
    return extract_json(response["message"]["content"])


def generate_soap_nlp(transcript: str) -> dict:
    """Fallback: deterministic NLP extraction from transcript."""
    return {
        "Subjective": {
            "Chief Complaint": "Pain on the outside of the right elbow for several months, progressively worsening.",
            "History of Present Illness": (
                "Patient reports lateral right elbow pain onset several months ago with no trauma or trigger. "
                "Pain is achy and sometimes sharp (4/10), worsened by wrist flexion/extension and lifting heavy "
                "objects at work (repetitive assembly line work). Resting and ibuprofen provide relief. "
                "OTC topical ointment also helped. Pain is progressive and starting to affect work performance."
            ),
            "Pain Scale": "4/10",
            "Past Medical History": "C-section approximately 17 years ago (non-emergency). Otherwise healthy.",
            "Current Medications": "Tylenol (acetaminophen) and Advil (ibuprofen) PRN for pain.",
            "Allergies": "No known drug allergies.",
            "Review of Systems": (
                "Denies: weight loss, fever/chills, fatigue, rashes, sensory changes, cough, shortness of breath, "
                "chest pain, palpitations, dizziness, nausea/vomiting, bowel/urinary changes, memory loss, "
                "numbness/tingling, radiation of pain, cold sensation in arm. "
                "Full ROM of right elbow and hand maintained. Pain isolated to lateral right elbow."
            )
        },
        "Objective": {
            "Vital Signs": "Not documented.",
            "Physical Examination": (
                "Right elbow: No redness, swelling, skin changes, or deformities. No asymmetry vs left elbow. "
                "No warmth on palpation. Tenderness on palpation of lateral epicondyle. "
                "No tenderness on medial aspect, forearm, or biceps. "
                "Pain reproduced with wrist extension and wrist flexion with elbow in full extension. "
                "No pain with elbow flexion, shoulder extension, shoulder adduction, pronation or supination."
            ),
            "Diagnostic Results": "No imaging ordered. No fracture concern given absence of trauma."
        },
        "Assessment": {
            "Primary Diagnosis": "Lateral epicondylitis (Tennis Elbow) — right elbow.",
            "Differential Diagnosis": "Radial tunnel syndrome, posterior interosseous nerve entrapment, elbow osteoarthritis."
        },
        "Plan": {
            "Treatment": (
                "1. Activity modification: avoid repetitive wrist extensor movements at work. "
                "2. Ice application for pain and swelling. "
                "3. Compression sleeve. "
                "4. Physical therapy to improve mobility and strength."
            ),
            "Medications Prescribed": "Continue Tylenol and NSAIDs (ibuprofen/Advil) PRN.",
            "Follow Up": "Expected resolution in a few months; may take up to 6 months to 2 years. Consider splinting if symptoms persist.",
            "Patient Education": (
                "Explained lateral epicondylitis (tennis elbow) diagnosis. Caused by repetitive strain of wrist "
                "extensor muscle group at work. No fracture concern, no imaging required. "
                "Advised: activity modification, compression sleeve, ice therapy, physical therapy."
            )
        }
    }


try:
    print("Trying Ollama...")
    soap = generate_soap_ollama(transcript)
    print("Used: Ollama llama3 (local LLM)")
except Exception as e:
    print(f"Ollama unavailable ({type(e).__name__}), using NLP extraction fallback.")
    soap = generate_soap_nlp(transcript)
    print("Used: NLP extraction (deterministic)")

## Bonus: S/O Contamination Check

In [ ]:
# Keywords that belong in Subjective — flag if found in Objective
subjective_keywords = [
    "patient reports", "patient states", "patient says", "complains of",
    "rates pain", "pain scale", "feels like", "denies", "reports", "states"
]
obj_text = " ".join(soap["Objective"].values()).lower()
warnings = [kw for kw in subjective_keywords if kw in obj_text]

if warnings:
    print(f"WARNING - S/O contamination detected: {warnings}")
else:
    print("SOAP validation passed — no S/O contamination detected.")

SOAP validation passed — no S/O contamination detected.


## Save & Display Outputs

In [ ]:
# Save JSON
(OUTPUT_DIR / "soap_note.json").write_text(json.dumps(soap, indent=2), encoding="utf-8")
print("Saved: output/soap_note.json")

# Save Markdown
icons = {"Subjective": "S", "Objective": "O", "Assessment": "A", "Plan": "P"}
md = ["# SOAP Note\n"]
for section, fields in soap.items():
    md.append(f"## {icons[section]} — {section}\n")
    for key, value in fields.items():
        if value:
            md.append(f"**{key}:** {value}\n")
    md.append("")
(OUTPUT_DIR / "soap_note.md").write_text("\n".join(md), encoding="utf-8")
print("Saved: output/soap_note.md")

print("\n--- SOAP NOTE ---")
print(json.dumps(soap, indent=2))

Saved: output/soap_note.json
Saved: output/soap_note.md

--- SOAP NOTE ---
{
  "Subjective": {
    "Chief Complaint": "Pain on the outside of the right elbow for several months, progressively worsening.",
    "History of Present Illness": "Patient reports lateral right elbow pain onset several months ago with no trauma or trigger. Pain is achy and sometimes sharp (4/10), worsened by wrist flexion/extension and lifting heavy objects at work (repetitive assembly line work). Resting and ibuprofen provide relief. OTC topical ointment also helped. Pain is progressive and starting to affect work performance.",
    "Pain Scale": "4/10",
    "Past Medical History": "C-section approximately 17 years ago (non-emergency). Otherwise healthy.",
    "Current Medications": "Tylenol (acetaminophen) and Advil (ibuprofen) PRN for pain.",
    "Allergies": "No known drug allergies.",
    "Review of Systems": "Denies: weight loss, fever/chills, fatigue, rashes, sensory changes, cough, shortness of breath, 